# Example of creating a custom Dataset for PyGeometric

See [docs](https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_dataset.html)


In [1]:
%load_ext autoreload
%autoreload 2

import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)
# Choose device (cuda if available, else cpu)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)

2.9.0+cu128
Selected device: cuda


# Example Custom Dataset Setup

Luckily, there is a new tutorial on how to include two graphs in one Data object

In [2]:
# Below is code produced by Gemini (Pro 2.5) to illustrate a pattern for how to achieve the batching we are looking for.
# NOTE: this seems easier/clearer than the official PyGeometric docs! Hopefully, this is correct :)
import torch
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader

# --- 1. Create some dummy graph data ---
# Let's assume you have two lists of graphs and a list of labels
graph_list_1 = []
graph_list_2 = []
labels = []

for _ in range(10): # Create 10 dummy data points
    # Graph 1
    num_nodes_1 = torch.randint(5, 10, (1,)).item()
    edge_index_1 = torch.randint(0, num_nodes_1, (2, num_nodes_1 * 2))
    x_1 = torch.randn(num_nodes_1, 8)
    graph_list_1.append(Data(x=x_1, edge_index=edge_index_1))

    # Graph 2
    num_nodes_2 = torch.randint(5, 10, (1,)).item()
    edge_index_2 = torch.randint(0, num_nodes_2, (2, num_nodes_2 * 2))
    x_2 = torch.randn(num_nodes_2, 8)
    graph_list_2.append(Data(x=x_2, edge_index=edge_index_2))

    # Label
    labels.append(torch.rand(1))


# --- 2. Define your Dataset class ---
# This is the key part for you
class GraphPairDataset(Dataset):
    def __init__(self, g_list_1, g_list_2, label_list):
        super().__init__()
        self.g_list_1 = g_list_1
        self.g_list_2 = g_list_2
        self.label_list = label_list

    def len(self):
        return len(self.label_list)

    def get(self, idx):
        # Just return the tuple
        data_1 = self.g_list_1[idx]
        data_2 = self.g_list_2[idx]
        label = self.label_list[idx]
        return data_1, data_2, label

# --- 3. Use the Dataset and DataLoader ---
dataset = GraphPairDataset(graph_list_1, graph_list_2, labels)
loader = DataLoader(dataset, batch_size=4)

# Get the first batch
batch = next(iter(loader))

In [3]:
batch

[DataBatch(x=[30, 8], edge_index=[2, 60], batch=[30], ptr=[5]),
 DataBatch(x=[26, 8], edge_index=[2, 52], batch=[26], ptr=[5]),
 tensor([[0.0893],
         [0.2388],
         [0.0989],
         [0.1146]])]

In [4]:
batch[0][3]

Data(x=[9, 8], edge_index=[2, 18])

In [5]:
batch[0].batch

tensor([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3,
        3, 3, 3, 3, 3, 3])

Cool, that seems to work! Now, let's try to adapt it to our specific data